## Setup Libraries

In [1]:
!pip install -q pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 6.8 MB/s eta 0:00:00


In [2]:
import random
import warnings
import numpy as np, pandas as pd
from colorama import Fore, Style
from importlib.metadata import version
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import torch
import pytabkit
from pytabkit import RealMLP_TD_Classifier

warnings.filterwarnings('ignore')
print("PyTorch  version:", torch.__version__)
print("PyTabKit version:", version("pytabkit"))

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
seed_everything(42)

PyTorch  version: 2.8.0+cu126
PyTabKit version: 1.7.3


## Load Data

In [3]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/test.csv")
orig = pd.read_csv("/kaggle/input/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety/EV_Adoption_and_Range_Anxiety_Dataset.csv")
print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Orig shape :", orig.shape)

Train shape: (668665, 15)
Test shape : (286571, 14)
Orig shape : (10000, 15)


## Preprocess Features

In [4]:
%%time
ID = 'id'
TARGET = 'Will_Buy_EV'
train[TARGET] = train[TARGET].map({'No': 0, 'Yes': 1})
orig[TARGET] = orig[TARGET].map({'No': 0, 'Yes': 1})
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]
del train, test
print("X      init shape:", X.shape)
print("X_test init shape:", X_test.shape, "\n")

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
important_combos = [
    ('Annual_Income_USD', 'Range_Anxiety_Level'),
    ('Age', 'Range_Anxiety_Level'),
    ('Annual_Income_USD', 'Income_/_100_floor_'),
]
def feature_engineering(df, fit=False):
    # Fill NaNs
    for col in cat_cols:
        df[col] = df[col].fillna("missing")
    for col in num_cols:
        df[col] = df[col].fillna(0.0)

    # Categorize string cats
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[col] = codes
        df[col] = df[col].astype('category')

    # Arithmetic interaction
    df['_Daily_Commute_km_/_Age'] = (df['Daily_Commute_km'] / (df['Age'] + 1e-6)).astype('float32')
    df['Income_/_100_floor_']  = np.floor(df['Annual_Income_USD'] / 100.0).astype('float32').astype('category')
    df['Income_/_1000_floor_']  = np.floor(df['Annual_Income_USD'] / 1000.0).astype('float32').astype('category')
    df['Income_/_10000_floor_']  = np.floor(df['Annual_Income_USD'] / 10000.0).astype('float32').astype('category')
    df['Daily_km_/_5_floor_']  = np.floor(df['Daily_Commute_km'] / 5.0).astype('float32').astype('category')

    # Categorize numericals
    for col in [i for i in num_cols if i not in ['Annual_Income_USD', 'Charging_Stations_Near_Home']]:
        cat_name = f"{col}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype('category')

    # Digit extraction
    for col in ['Daily_Commute_km']:
        decimal_name = f"_{col}_decimal"
        df[decimal_name] = (df[col] % 1).round(2).astype('float32')
    df['Annual_Income_USD_is_multiple_10_'] = (np.floor(df['Annual_Income_USD']) % 10 == 0).astype('category')

    # Target encoding from orig
    for col in ['Annual_Income_USD']:
        orig_enc_name = f"_{col}_mean_target_orig"
        df[orig_enc_name] = (
            df[col]
            .map(orig.groupby(col)[TARGET].mean())
            .fillna(orig[TARGET].mean())
            .astype('float32')
        )

    # Count encoding
    for col in ['Annual_Income_USD']:
        count_name = f"_{col}_count"
        if fit:
            count_map = df[col].value_counts()
            category_map[count_name] = count_map
        else:
            count_map = category_map[count_name]
        df[count_name] = df[col].astype(object).map(count_map).fillna(0).astype('int32')

    # Discretize numericals
    bin_config = {'Annual_Income_USD': [400, 600, 800, 900, 1100]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ['quantile']:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"
                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode='ordinal',
                        strategy=strategy,
                        subsample=None
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = binned
                df[bin_name] = df[bin_name].astype('category')

    # Create interaction categories
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype('category')   

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)
cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X      prep shape:", X.shape)
print("X_test prep shape:", X_test.shape, "\n")

X      init shape: (668665, 13)
X_test init shape: (286571, 13) 

init len(cat_cols): 6
init len(num_cols): 7 

len(new_cat_cols): 18
len(new_num_cols): 4 

prep len(cat_cols): 24
prep len(num_cols): 11 

X      prep shape: (668665, 35)
X_test prep shape: (286571, 35) 

CPU times: user 4.64 s, sys: 691 ms, total: 5.33 s
Wall time: 5.36 s


## Config

In [5]:
class CFG:
    FOLDS = 5
    SEED = 42
    TE = True

params = {
    'random_state': 42,
    'verbosity': 2,
    'val_metric_name': '1-auc_ovr',

    'n_ens': 8,
    'n_epochs': 3,
    'batch_size': 256,
    'use_early_stopping': False,
    'early_stopping_additive_patience': 10,
    'early_stopping_multiplicative_patience': 1,

    'lr': 0.06,
    'wd': 0.0236,
    'sq_mom': 0.988,
    'lr_sched': 'lin_cos_log_15',
    'first_layer_lr_factor': 0.25,

    'embedding_size': 5,
    'max_one_hot_cat_size': 18,
    'hidden_sizes': [512, 256, 128],
    'act': 'silu',
    'p_drop': 0.05,
    'p_drop_sched': 'expm4t',

    'plr_hidden_1': 16,
    'plr_hidden_2': 8,
    'plr_act_name': 'gelu',
    'plr_lr_factor': 0.1151,
    'plr_sigma': 2.33,

    'ls_eps': 0.01,
    'ls_eps_sched': 'sqrt_cos',

    'add_front_scale': False,
    'bias_init_mode': 'neg-uniform-dynamic-2',
    'tfms': ['one_hot', 'median_center', 'robust_scale',
             'smooth_clip', 'embedding', 'l2_normalize'],
}

## Train K-Fold

In [6]:
%%time

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_tr = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]
    X_tst = X_test.copy()  

    # Target encoding
    if CFG.TE:
        te_cols = combo_names
        TE = TargetEncoder(cv=5, smooth='auto', shuffle=True, random_state=42)
        tr_enc = TE.fit_transform(X_tr[te_cols], y_tr)
        val_enc = TE.transform(X_val[te_cols])
        tst_enc = TE.transform(X_tst[te_cols])

        te_names = [f"_{col}TE" for col in te_cols]
        X_tr[te_names] = tr_enc
        X_val[te_names] = val_enc
        X_tst[te_names] = tst_enc

    if fold == 1: print("len(FEATURES):", len(X_tr.columns.tolist()), "\n")
    print("#"*16)
    print(f"### Fold {fold}/{CFG.FOLDS} ...")
    print("#"*16)

    model = RealMLP_TD_Classifier(**params)
    model.fit(X_tr, y_tr, X_val, y_val)

    val_preds = model.predict_proba(X_val)[:, 1]
    fold_test_preds = model.predict_proba(X_tst)[:, 1]

    oof_preds[val_idx] = val_preds
    test_preds += fold_test_preds / CFG.FOLDS

    fold_score = roc_auc_score(y_val, val_preds)
    print(f"{Fore.GREEN}{Style.BRIGHT}Fold {fold} | Score: {fold_score:.5f}{Style.RESET_ALL}\n")
    torch.cuda.empty_cache()

len(FEATURES): 38 

################
### Fold 1/5 ...
################
Columns classified as continuous: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', '_Daily_Commute_km_/_Age', '_Daily_Commute_km_decimal', '_Annual_Income_USD_mean_target_orig', '_Annual_Income_USD_count', '_Annual_Income_USD_Range_Anxiety_Level_TE', '_Age_Range_Anxiety_Level_TE', '_Annual_Income_USD_Income_/_100_floor__TE']
Columns classified as categorical: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Income_/_100_floor_', 'Income_/_1000_floor_', 'Income_/_10000_floor_', 'Daily_km_/_5_floor_', 'Age_cat_', 'Daily_Commute_km_cat_', 'Number_of_Cars_Owned_cat_', 'Charging_Stations_Near_Work_cat_', 'Environmental_Concern_Level_cat_', 'Annual_Income_USD_is_multiple_10_', 'Annual_Income_USD_400_quantile_bin_', 'Annual_Income_USD_600_quan

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/3: val 1-auc_ovr = 0.055683
Epoch 2/3: val 1-auc_ovr = 0.055438
Epoch 3/3: val 1-auc_ovr = 0.054873


`Trainer.fit` stopped: `max_epochs=3` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 1 | Score: 0.94513

################
### Fold 2/5 ...
################
Columns classified as continuous: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', '_Daily_Commute_km_/_Age', '_Daily_Commute_km_decimal', '_Annual_Income_USD_mean_target_orig', '_Annual_Income_USD_count', '_Annual_Income_USD_Range_Anxiety_Level_TE', '_Age_Range_Anxiety_Level_TE', '_Annual_Income_USD_Income_/_100_floor__TE']
Columns classified as categorical: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Income_/_100_floor_', 'Income_/_1000_floor_', 'Income_/_10000_floor_', 'Daily_km_/_5_floor_', 'Age_cat_', 'Daily_Commute_km_cat_', 'Number_of_Cars_Owned_cat_', 'Charging_Stations_Near_Work_cat_', 'Environmental_Concern_Level_cat_', 'Annual_Income_USD_is_multiple_10_', 'Annual_Income_USD_400_quantile_bin_', 'Annual_Income_USD_600

...

Fold 9 | Score: 0.94714

################
### Fold 10/5 ...
################
Columns classified as continuous: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', '_Daily_Commute_km_/_Age', '_Daily_Commute_km_decimal', '_Annual_Income_USD_mean_target_orig', '_Annual_Income_USD_count', '_Annual_Income_USD_Range_Anxiety_Level_TE', '_Age_Range_Anxiety_Level_TE', '_Annual_Income_USD_Income_/_100_floor__TE']
Columns classified as categorical: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Income_/_100_floor_', 'Income_/_1000_floor_', 'Income_/_10000_floor_', 'Daily_km_/_5_floor_', 'Age_cat_', 'Daily_Commute_km_cat_', 'Number_of_Cars_Owned_cat_', 'Charging_Stations_Near_Work_cat_', 'Environmental_Concern_Level_cat_', 'Annual_Income_USD_is_multiple_10_', 'Annual_Income_USD_400_quantile_bin_', 'Annual_Income_USD_60

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/3: val 1-auc_ovr = 0.055442
Epoch 2/3: val 1-auc_ovr = 0.055553
Epoch 3/3: val 1-auc_ovr = 0.054833


`Trainer.fit` stopped: `max_epochs=3` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 10 | Score: 0.94517

CPU times: user 20min 19s, sys: 8.57 s, total: 20min 28s
Wall time: 20min 1s


## Evaluation and Submission

In [7]:
oof_score = roc_auc_score(y, oof_preds)
print("\n" + "="*26)
print(f"Overall OOF Score: {Fore.BLACK}{Style.BRIGHT}{oof_score:.6f}{Style.RESET_ALL}")
print("="*26)

# oof_df = pd.DataFrame({ID: train_id, TARGET: oof_preds})
# oof_df.to_csv('oof_preds.csv', index=False)

np.save(f"oof_pytab_REALMLP_{oof_score:.6f}.npy", oof_preds)
np.save(f"test_pytab_REALMLP_{oof_score:.6f}.npy", test_preds)

sub = pd.DataFrame({ID: test_id, TARGET: test_preds})
sub.to_csv('submission.csv', index=False)
sub.head()


Overall OOF Score: 0.946139


,id,Will_Buy_EV
0,668665,0.060491
1,668666,0.029810
2,668667,0.006307
3,668668,0.005330
4,668669,0.062377
